# IPython magics

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

SciQLop ships several IPython magics that turn common tasks into one-liners.

**What you'll learn**
- `%%vp` — define a virtual product in a single cell, with type annotations, debug mode, and hot-reload.
- `%plot` — quickly plot a product by fuzzy-matched name.
- `%timerange` — read or set the time range of plot panels.
- Where the other magics (`%%layer`, `%job`, `%install`, `%workspace`) are covered.

**Prerequisites** — [Plot panels](5-SciQLopBasicPlotPanel.ipynb).

**Next** — [Virtual products](6-SciQLopVirtualProducts.ipynb), [Parameterised virtual products (knobs)](15-SciQLopParameterizedVirtualProducts.ipynb).


## 1. The `%%vp` cell magic

The `%%vp` magic lets you define and register a virtual product in a single cell.
Just write a function — the magic handles registration, type inference, and hot-reload on re-execution.

A cell may also contain **private helper functions** whose names start with `_`: they are ignored by the registration, and exactly one public function must remain — that one becomes the virtual product.

### 1.1 Scalar product with type annotation

Use a return type annotation like `-> Scalar["label"]` to declare the product type explicitly.
The annotation types (`Scalar`, `Vector`, `MultiComponent`, `Spectrogram`) are automatically available in `%%vp` cells.

In [ ]:
%%vp --path "examples/sine_wave"
import numpy as np

def sine_wave(start: float, stop: float) -> Scalar["sin"]:
    x = np.arange(start, stop, 5.0)
    return x, np.sin(x / 100)

### 1.2 Vector product

For a 3-component vector, annotate with `-> Vector["x", "y", "z"]`.

In [ ]:
%%vp --path "examples/rotating_vector"
import numpy as np

def rotating_vector(start: float, stop: float) -> Vector["Vx", "Vy", "Vz"]:
    t = np.arange(start, stop, 5.0)
    vx = np.cos(t / 200)
    vy = np.sin(t / 200)
    vz = np.zeros_like(t)
    return t, np.column_stack([vx, vy, vz])

### 1.3 Type inference (no annotation)

If you omit the return annotation, `%%vp` will call your function once and infer the type from the returned data shape.
This is convenient for quick prototyping, but explicit annotations are recommended.

In [ ]:
%%vp --path "examples/inferred"
import numpy as np

def inferred_product(start: float, stop: float):
    t = np.arange(start, stop, 5.0)
    return t, np.cos(t / 50) * np.exp(-t / 10000)

### 1.4 Debug mode

Add `--debug` to get a diagnostic panel showing execution time, data shape, and error traces.
Very useful when developing a new virtual product.

In [ ]:
%%vp --debug --path "examples/debugged"
import numpy as np

def debugged_product(start: float, stop: float) -> Scalar["value"]:
    t = np.arange(start, stop, 5.0)
    return t, np.sin(t / 100) + 0.1 * np.random.randn(len(t))

### 1.5 Hot-reload

Re-executing a `%%vp` cell updates the callback in-place — any open plot showing
that product will use the new function on the next data request. No need to close and re-plot.

### 1.6 More options

- `--cachable` promises that the function returns the same full-resolution data for a given interval, whatever range is requested. SciQLop then fetches twice the visible window and serves pans and zooms inside it without calling the function again. Leave it off for a function that returns a fixed number of points per request. Example: `%%vp --cachable --path "examples/sine_wave"`.
- `--start` / `--stop` set the time range used by `--debug`, and by the single test call that infers the type when there is no return annotation.
- Return annotations written under `from __future__ import annotations` are read too.

## 2. The `%plot` line magic

`%plot` lets you plot a product by name without writing Python code.
It supports fuzzy matching — you don't need the full product path.

In [ ]:
# Plot by product path — creates a new panel
%plot examples/sine_wave

In [ ]:
# Plot into a specific panel (use the panel name shown in the tab title)
# %plot examples/rotating_vector Panel0

Panel names are the tab titles: `Panel0`, `Panel1`, … unless you renamed the panel. Names with spaces must be quoted.

## 3. The `%timerange` line magic

Inspect or set the time range of plot panels.

In [ ]:
# Print time ranges of all open panels
%timerange

In [ ]:
# Set time range on a specific panel
# %timerange "2020-01-01" "2020-01-02" Panel0

## 4. Other magics

- `%%layer` — define a reactive annotation layer (markers, spans, horizontal lines) in one cell. See [Annotation layers](16-SciQLopAnnotationLayers.ipynb).
- `%job` — run a shell command as a background job and follow it: `%job submit <command>`, `%job list`, `%job status <id>`, `%job cancel <id>`.
- `%install` and `%workspace` — manage the workspace's packages and examples. See [Workspaces](11-SciQLopWorkspace.ipynb).

## Summary

| Magic | Purpose | Example |
|-------|---------|--------|
| `%%vp` | Define a virtual product | `%%vp --path "my/product"` |
| `%plot` | Quick-plot with fuzzy match | `%plot mms1_fgm` |
| `%timerange` | Get/set panel time range | `%timerange 2020-01-01 2020-01-02 Panel0` |
| `%%layer` | Define an annotation layer | `%%layer --path "my/layer"` |
| `%job` | Run and follow background jobs | `%job list` |
| `%install` / `%workspace` | Manage workspace packages and examples | see [Workspaces](11-SciQLopWorkspace.ipynb) |